# Exploring a Reasoning ("Thinking") Model

Some models are trained to **reason step-by-step before answering**. Models like `deepseek-ai/DeepSeek-R1-Distill-Llama-8B` emit an explicit chain-of-thought wrapped in `<think>...</think>` and then give the final answer — the same behavior we *taught* from scratch in the GRPO notebook. Here we simply observe it in a model that already has it.

This notebook:

1. Loads the distilled reasoning model.
2. Generates an answer and prints the **raw output including special tokens**, so you can see the `<think>` block and the model's chat markers.
3. Shows how the chat template wraps a reasoning turn.

> **Requirements:** a GPU runtime and a Hugging Face token (loaded from `.env`). Outputs below were captured from a real run.

## 1. Setup & authentication

In [ ]:
# --- Install dependencies --- (+ python-dotenv to read the HF token)
!pip install -q -U transformers datasets bitsandbytes trl peft huggingface_hub python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load the Hugging Face token from a local .env file (see .env.example):
#   HF_TOKEN=hf_your_token_here
load_dotenv()

hf_token = os.getenv("HF_TOKEN")
if not hf_token or hf_token == "your_hugging_face_token_here":
    raise ValueError(
        "HF_TOKEN not found. Create a .env file with HF_TOKEN=hf_your_token_here "
        "(get a token at https://huggingface.co/settings/tokens)."
    )

# Authenticate so we can download gated models and push results to the Hub.
login(token=hf_token)
print("Successfully authenticated with the Hugging Face Hub.")

Successfully authenticated with the Hugging Face Hub.


### The reasoning-model prompt/response format

A reasoning model wraps its chain-of-thought in `<think>...</think>` before the final answer:

```
### Instruction:
You are a helpful assistant.

### Question: (User Question)
{ User input }

### Response: (Model Answer)
<think>
{ Reasoning steps (Chain of Thought) }
</think>
{ Final answer }
```

## 2. Generate with a reasoning model

Watch the model produce a `<think>` chain-of-thought before its final answer.

In [ ]:
# --- Load a distilled REASONING model ---
# DeepSeek-R1-Distill-Llama-8B is trained to 'think out loud': it emits its
# chain-of-thought inside <think>...</think> before giving the final answer.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig

model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",  # Let Transformers automatically handle device placement
)

# Convert the chat messages into input token ids
messages = [{"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is fine tuning? Explain clearly please."}

]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)

# Generate output token ids from the model
outputs = model.generate(
    inputs,
    max_new_tokens=1024,
    return_dict_in_generate=True,  # makes the generated token ids accessible
    output_scores=True
)

# Take the raw generated token ids
generated_token_ids = outputs.sequences[0]

# Decode the tokens back to text (including special tokens)
decoded_text = tokenizer.decode(generated_token_ids, skip_special_tokens=False)

print("Raw output (including special tokens):")
print(decoded_text)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Raw output (including special tokens):
<｜begin▁of▁sentence｜>You are a helpful assistant.<｜User｜>What is fine tuning? Explain clearly please.<think>
Okay, so I need to figure out what fine tuning is. I've heard the term before, especially in contexts like machine learning or AI, but I'm not entirely sure what it exactly means. Let me try to break it down.

First, I think fine tuning has something to do with adjusting something small or making precise changes. Like, when you tune a guitar, you make small adjustments to the strings or tuning pegs to get the right pitch. So maybe fine tuning in a broader sense is making small adjustments to something complex to make it work better.

In the context of machine learning models, I've heard people talk about fine-tuning pre-trained models. So, maybe a model that's already been trained on a lot of data is fine-tuned to fit a specific task or dataset. But how exactly does that work?

I remember reading that models like BERT are pre-trained on lar

In [ ]:
# Print the full decoded output again (reasoning + final answer).
print(decoded_text)

<｜begin▁of▁sentence｜>You are a helpful assistant.<｜User｜>What is fine tuning? Explain clearly please.<think>
Okay, so I need to figure out what fine tuning is. I've heard the term before, especially in contexts like machine learning or AI, but I'm not entirely sure what it exactly means. Let me try to break it down.

First, I think fine tuning has something to do with adjusting something small or making precise changes. Like, when you tune a guitar, you make small adjustments to the strings or tuning pegs to get the right pitch. So maybe fine tuning in a broader sense is making small adjustments to something complex to make it work better.

In the context of machine learning models, I've heard people talk about fine-tuning pre-trained models. So, maybe a model that's already been trained on a lot of data is fine-tuned to fit a specific task or dataset. But how exactly does that work?

I remember reading that models like BERT are pre-trained on large datasets, and then people fine-tune 

## 3. How the chat template wraps a reasoning turn

In [ ]:
# Provide the assistant turn ourselves to see how the chat template wraps a
# reasoning conversation with the model's special tokens.
messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is fine tuning? Explain clearly please."},
            {"role": "assistant", "content": "<think>Okay, so I need to figure out what fine tuning is.</think> Fine-tuning is a process....."},
            ]


tokenizer.apply_chat_template(messages, tokenize=False)

'<｜begin▁of▁sentence｜>You are a helpful assistant.<｜User｜>What is fine tuning? Explain clearly please.<｜Assistant｜> Fine-tuning is a process.....<｜end▁of▁sentence｜>'